## 1. Read Files From Raw Files (Clinical data, Transcriptomic data, Methylomics data, Metabolomics data)

1.1 Read Clinical data

In [ ]:
import pandas as pd 
phenodata_df = pd.read_excel('./data/pheno_data/LLFS_phenos_21JUN2022.xlsx', sheet_name='Phenodata').sort_values(by='subject')
# Convert the subject column to string
phenodata_df['subject'] = phenodata_df['subject'].astype(str)
display(phenodata_df)

In [ ]:
### 读取t2ds标签数据
import numpy as np
# 从指定的 txt 文件中读取表格数据
t2ds_label_df = pd.read_table('./data/label_data/t2dpret2d.txt')
# 将表格中的'.'符号替换为0
t2ds_label_df = t2ds_label_df.replace('.', 0)
# 将 pret2ds 列转换为 np.int64 类型（整型）
t2ds_label_df['pret2ds'] = t2ds_label_df['pret2ds'].astype(np.int64)
# 按照 subject 列进行排序
t2ds_label_df = t2ds_label_df.sort_values(by='subject')
# 将 subject 列值转换为字符串类型
t2ds_label_df['subject'] = t2ds_label_df['subject'].astype(str)
# 打印每一列的数据类型
print(t2ds_label_df.dtypes)
# 展示处理后的数据
display(t2ds_label_df)

### 1.2 Transcriptomic Data

In [ ]:
# 读取转录组残差数据，并按 subject 排序
tran_v1_df = pd.read_csv('./data/omics_data/residuals/RNA_seq_residuals_v1_allsubjects.csv').sort_values(by='subject')
display(tran_v1_df)

# 转置表格，使基因/样本切换位置
tran_v1_df_transposed = tran_v1_df.T

# 将第一行为字符串，去除末尾“.0”，设为新的表头
tran_v1_df_transposed.columns = tran_v1_df_transposed.iloc[0].astype(str).str.replace('.0', '', regex=False)

# 删除已作为表头的首行
tran_v1_df_transposed = tran_v1_df_transposed.drop(tran_v1_df_transposed.index[0])

# 重置索引
tran_v1_df = tran_v1_df_transposed.reset_index()

# 将第一列 'index' 重命名为 'gene_id'
tran_v1_df = tran_v1_df.rename(columns={'index': 'gene_id'})

# 将版本号基因ID转换为标准基因ID（去除点号后的内容）
ensembl_gene_ids = tran_v1_df['gene_id'].apply(lambda x: x.split('.')[0]).tolist()
tran_v1_df['gene_id'] = ensembl_gene_ids

# 展示处理后的数据表
display(tran_v1_df)

In [ ]:
# Keep the gene ID in the ensembl_data dataframe
ensembl_data = pd.read_csv('./data/kg_data/ensembl/mart_export_genename.txt')
ensembl_data= ensembl_data.rename(columns={'Gene stable ID': 'gene_id'}).dropna().drop_duplicates().reset_index(drop=True)
display(ensembl_data)
# Convert the 'Gene Name' column to 'gene_name'
ensembl_data = ensembl_data.rename(columns={'Gene name': 'gene_name'})
merged_tran_v1_df = pd.merge(tran_v1_df, ensembl_data, on='gene_id', how='inner')
# Move the gene name to the first column
merged_tran_v1_df = merged_tran_v1_df[['gene_name'] + [col for col in merged_tran_v1_df.columns if col != 'gene_name']]
# Drop the gene ID column
merged_tran_v1_df = merged_tran_v1_df.drop(columns=['gene_id'])
# Drop duplicated rows and aggregate the rows by grouping them by gene name
merged_tran_v1_df = merged_tran_v1_df.groupby(['gene_name']).mean().reset_index()
display(merged_tran_v1_df)